# Task 1 · Part 2 — From the LLM to the classifier

**Guiding question:** Can Peruvian news headlines be transformed into a reliable news classifier?

Building upon the dataset constructed in Part 1, this notebook compares three increasingly specialized approaches to headline classification: a zero-shot model that requires no training, a general-purpose large language model (LLM) used as a weak annotator, and a lightweight Transformer fine-tuned on the collected headlines. Accordingly, the notebook proceeds from the least to the most task-specific approach, so that each stage can be assessed against the preceding one.

**Methodological note on the reference labels.** Throughout the notebook, the section-based labels assigned in Part 1 (`etiqueta`) serve as the reference labels, given that they derive from the editorial decisions of El Comercio rather than from an automated procedure. The LLM, by contrast, is treated as a weak annotator whose output is audited and evaluated against these reference labels.

In [7]:
import os
import json
import time
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from google import genai
from google.genai import types
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import pipeline

This cell loads the libraries required throughout the notebook. Specifically, `transformers` provides access to pretrained language models, whereas `google-genai` connects to the Gemini LLM; in turn, `dotenv` reads the API key from a local file that is excluded from the repository, thereby preventing credentials from being exposed. Furthermore, `scikit-learn` supplies the stratified data split and the evaluation metrics, while `seaborn` and `matplotlib` render the confusion matrix.

In [8]:
# =============================================
#  CONFIGURATION
# =============================================

# The API key is read from the local .env file, which Git ignores.
load_dotenv()

# Fixed seed so that every random step can be reproduced exactly.
semilla = 42

categorias = ["politica", "economia", "deportes", "ciencia_tecnologia"]

noticias = pd.read_csv("datos/noticias.csv")
print(noticias.shape)
noticias["etiqueta"].value_counts()

(383, 7)


etiqueta
economia              99
deportes              98
ciencia_tecnologia    95
politica              91
Name: count, dtype: int64

The configuration cell loads the API key from the `.env` file, defines a fixed random seed (`42`) to guarantee the reproducibility of every stochastic procedure, and lists the four official categories. Subsequently, the dataset exported in Part 1 is loaded; its dimensions and category distribution confirm that the 383 news items are available for the classification stage.

## 1. Zero-shot baseline

Before any training, a zero-shot classifier establishes the minimum performance that subsequent approaches must exceed. Such a model, pretrained on natural language inference, evaluates how plausibly a headline entails a hypothesis of the form *"This news item is about {category}"*, and thus assigns categories it was never explicitly trained on. Following the assignment, 40 headlines are evaluated; to preserve the balance between classes, ten headlines are sampled from each category.

In [9]:
# Ten headlines per category, sampled with the fixed seed.
muestra_zero_shot = noticias.groupby("etiqueta").sample(n=10, random_state=semilla)
muestra_zero_shot["etiqueta"].value_counts()

etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64

A stratified sample of 40 headlines is drawn, with exactly ten items per category. Stratification prevents the baseline from being distorted by an over-representation of any single category, while the fixed seed ensures that the same sample is obtained in every execution.

In [10]:
# Spanish model trained on natural language inference (lighter than XLM-RoBERTa-large).
clasificador_zero_shot = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
)

# The model reads Spanish, so the candidate labels are written in Spanish.
etiquetas_candidatas = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia",
}

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The baseline relies on `Recognai/bert-base-spanish-wwm-cased-xnli`, one of the two models suggested by the assignment. It was preferred over `joeddav/xlm-roberta-large-xnli` because it is trained specifically on Spanish and is considerably lighter, which makes it feasible to run on a standard CPU. Since the model processes Spanish text, the candidate labels are expressed in Spanish and subsequently mapped back onto the official category names.

In [11]:
predicciones_zero_shot = []

for titular in muestra_zero_shot["titular"]:
    resultado = clasificador_zero_shot(
        titular,
        candidate_labels=list(etiquetas_candidatas.keys()),
        hypothesis_template="Esta noticia trata sobre {}.",
    )
    # The first label returned is the one with the highest score.
    predicciones_zero_shot.append(etiquetas_candidatas[resultado["labels"][0]])

muestra_zero_shot["prediccion_zero_shot"] = predicciones_zero_shot

accuracy_zero_shot = accuracy_score(muestra_zero_shot["etiqueta"], muestra_zero_shot["prediccion_zero_shot"])
f1_zero_shot = f1_score(muestra_zero_shot["etiqueta"], muestra_zero_shot["prediccion_zero_shot"], average="macro")

print(f"Zero-shot accuracy : {accuracy_zero_shot:.2f}")
print(f"Zero-shot macro F1 : {f1_zero_shot:.2f}")

Zero-shot accuracy : 0.62
Zero-shot macro F1 : 0.59


Without any task-specific training, the zero-shot model reaches an accuracy of 0.62 and a macro F1 of 0.59 on the 40 sampled headlines. Although this performance clearly exceeds the 0.25 expected from random guessing across four categories, it remains modest: roughly four out of ten headlines are misclassified. This limitation is consistent with the nature of the task, since headlines are short and frequently rely on proper names or context-dependent references rather than on explicit thematic vocabulary. Accordingly, a macro F1 of 0.59 constitutes the baseline that the subsequent approaches must exceed.

## 2. Weak supervision with an LLM

In the second stage, a general-purpose LLM is employed as a weak annotator for the entire dataset. Rather than relying on manual annotation, which is costly, the LLM is guided by a structured prompt that defines each category explicitly and requests a machine-readable output. To reduce the number of requests and remain within the free usage tier, headlines are submitted in batches of twenty.

The prompt is structured in three components. First, it assigns the model a role (an expert news editor); second, it defines each category through representative topics, which reduces ambiguity at the boundaries between classes; third, it prescribes a strict JSON output that includes a confidence score. This structure ensures that the responses can be parsed automatically and aligned with the original headlines through their identifiers.

In [12]:
cliente_gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# The lighter Flash-Lite model was used because the standard Flash model was saturated.
modelo_llm = "gemini-flash-lite-latest"

instrucciones = """You are an expert news editor. Classify each Peruvian news headline into exactly one of these four categories:
- politica: government, Congress, elections, political parties, public institutions, justice.
- economia: business, markets, prices, employment, trade, public finance, personal finance.
- deportes: sports, athletes, teams, competitions, sports events.
- ciencia_tecnologia: science, technology, artificial intelligence, digital platforms, environment, scientific research.

Return ONLY a JSON list with one object per headline, in the same order, using this format:
[{"id": 0, "etiqueta": "politica", "confianza": 0.95}]
where "confianza" is your confidence between 0 and 1."""

The prompt is structured in three components. First, it assigns the model a role (an expert news editor); second, it defines each category through representative topics, which reduces ambiguity at the boundaries between classes; third, it prescribes a strict JSON output that includes a confidence score. This structure ensures that the responses can be parsed automatically and aligned with the original headlines through their identifiers. Notably, the Flash-Lite version of Gemini was adopted after the standard Flash model repeatedly returned `503 UNAVAILABLE` errors due to high demand.

In [13]:
lista_etiquetas_llm = []
lotes_fallidos = []
tamano_lote = 20

for inicio in range(0, len(noticias), tamano_lote):
    lote = noticias.iloc[inicio:inicio + tamano_lote]

    titulares_numerados = ""
    for posicion, titular in enumerate(lote["titular"]):
        titulares_numerados += f"{posicion}. {titular}\n"

    # Up to four attempts, since the free tier may be temporarily saturated.
    resultados = []
    for intento in range(4):
        try:
            respuesta = cliente_gemini.models.generate_content(
                model=modelo_llm,
                contents=instrucciones + "\n\nHeadlines:\n" + titulares_numerados,
                config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0),
            )
            resultados = json.loads(respuesta.text)
            break
        except Exception as error:
            print("Retrying batch", inicio, "->", str(error)[:80])
            time.sleep(30 * (intento + 1))

    # Failed batches are recorded instead of being silently skipped.
    if len(resultados) == 0:
        lotes_fallidos.append(inicio)

    for resultado in resultados:
        fila = lote.iloc[resultado["id"]]
        lista_etiquetas_llm.append({
            "indice": fila.name,
            "titular": fila["titular"],
            "etiqueta_llm": resultado["etiqueta"],
            "confianza": resultado["confianza"],
            "modelo_llm": modelo_llm,
            "fecha": date.today().isoformat(),
        })

    print(inicio + len(lote), "headlines processed")
    # Pause between batches to respect the free-tier rate limit.
    time.sleep(7)

print("Failed batches:", lotes_fallidos)

20 headlines processed
40 headlines processed
60 headlines processed
80 headlines processed
100 headlines processed
120 headlines processed
140 headlines processed
160 headlines processed
180 headlines processed
200 headlines processed
220 headlines processed
240 headlines processed
260 headlines processed
280 headlines processed
300 headlines processed
320 headlines processed
340 headlines processed
360 headlines processed
380 headlines processed
383 headlines processed
Failed batches: []


The loop submits the headlines in batches of twenty, each numbered so that every response can be matched to its original headline. Setting the temperature to zero makes the model's answers as deterministic as possible, which favors reproducibility. Furthermore, each batch is attempted up to four times, with progressively longer waits, and any batch that ultimately fails is recorded rather than silently discarded. A pause also separates consecutive requests in order to comply with the rate limits of the free tier. Finally, each result is stored together with the model name and the labeling date, as required by the assignment.

In [14]:
etiquetas_llm = pd.DataFrame(lista_etiquetas_llm)

# Labels outside the four official categories are treated as invalid.
etiquetas_llm.loc[~etiquetas_llm["etiqueta_llm"].isin(categorias), "etiqueta_llm"] = None

etiquetas_llm[["titular", "etiqueta_llm", "confianza", "modelo_llm", "fecha"]].to_csv(
    "datos/etiquetas_llm.csv", index=False
)

print(etiquetas_llm.shape)
print("Invalid labels:", etiquetas_llm["etiqueta_llm"].isna().sum())
etiquetas_llm["etiqueta_llm"].value_counts()

(383, 6)
Invalid labels: 0


etiqueta_llm
politica              103
deportes               98
ciencia_tecnologia     92
economia               90
Name: count, dtype: int64

The LLM annotations are gathered into a DataFrame, and any label falling outside the four official categories is flagged as invalid rather than silently accepted. The file `datos/etiquetas_llm.csv` is then exported with the five columns required by the assignment: `titular`, `etiqueta_llm`, `confianza`, `modelo_llm` and `fecha`.